&nbsp;

&nbsp;

# Diff Analysis Result Visualization

- 図の可視化ではなく、表の作成を主に行う

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import xlsxwriter
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display
from matplotlib.colors import to_rgba
from matplotlib.ticker import MultipleLocator


os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261108


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Diff Analysis (before up5p transcripts - CRE integration)

- Diff Analysis で同定された CRE に関するデータを整備する

&nbsp;

&nbsp;

&nbsp;

## 1-1. MetaData

- どのペアが、BEDファイルとして出力されたか、そのペアがどの SRX ID で構成されているのかを表にする
- 1 行 = BED ファイルまで出力された 1 データセット (ATAC-seq: 10、HSF1 ChIP-seq: 6、ヒストン ChIP-seq: 2)
    - そもそも Diff Analysis で結果が得られなかったペア (ATAC-seq の `pair9`、HSF1 の `pair4`, `pair5`) は表に含まれない
    - `pair9` (GSE267931) は、`SRX24613861` に hg38 の bigWig / BED が存在せず、Diff Analysis が実行できなかったため除外 ([../../config/chipatlas_diff/atacseq/atacseq_hs_diff_scatter.yml](../../config/chipatlas_diff/atacseq/atacseq_hs_diff_scatter.yml))

&nbsp;

- 各カラムの内容
    - `diff_dataset`: bedtools intersect の出力ファイル名から取得したデータセット名
    - `sample`, `temperature`: 手入力用の空欄
    - `dataset_A_srx`, `dataset_B_srx`: [../../config/chipatlas_diff/](../../config/chipatlas_diff/) の `dataset_A/` (heat stress)、`dataset_B/` (control) に記載した SRX ID を `,` で連結したもの
    - `has_significant_dpr`: シスエレメントと重なる DPR のうち、q-value < 0.01 を満たすものが 1 つ以上あれば `True`

&nbsp;

- `has_significant_dpr` は log2 (FC) で絞る前の判定のため、`only_cre/` の parquet (log2 (FC) > 0.5 適用後) ではなく、`bedfile_intersect_v2/` の BED ファイルから直接判定している
- ATAC-seq は、ジョブ名 (`result_basename`) と BED ファイル名が異なるため、ペアと BED ファイルの対応をコード内に明記している
    - 各ジョブの結果ファイル (`wabi_result.bed`) とリネーム後の BED ファイルのチェックサムを照合して対応を確認済み
    - HSF1 は BED ファイル名に `pairN` が含まれるため、対応表は不要
- ヒストン ChIP-seq は、`dataset_A/` のファイル名が BED ファイル名と一致し、同じペアの A / B ファイルが `GSEnnn_chip_<抗体>` の接頭辞を共有するため、これを使って対応づけている

In [3]:
def read_srx_lists(config_dir: str) -> pl.DataFrame:
    """
    One row per pair, with comma-joined SRX IDs of dataset A (heat stress) and B (control).
    """
    return (
        pl.scan_csv(
            sorted(pathlib.Path(config_dir).glob("dataset_*/*.txt")),
            has_header=False,
            new_columns=["srx_id"],
            include_file_paths="path",
        )
        .with_columns(
            pl.col("path").str.extract(r"(dataset_[AB])").alias("dataset"),
            pl.col("path").str.extract(r"/(pair\d+)_").alias("pair"),
        )
        .group_by("pair", "dataset", maintain_order=True)
        .agg(pl.col("srx_id").str.join(","))
        .collect()
        .pivot(on="dataset", index="pair", values="srx_id")
        .rename({"dataset_A": "dataset_A_srx", "dataset_B": "dataset_B_srx"})
    )

In [4]:
def read_dpr_flag(bed_dir: str, q_threshold: float = 0.01) -> pl.DataFrame:
    """
    One row per bedtools intersect output (= Diff Analysis run that produced a BED file).
    has_significant_dpr is True if at least one CRE-overlapping DPR has q-value < q_threshold.
    """
    return (
        pl.scan_csv(
            sorted(pathlib.Path(bed_dir).glob("*_intersecting_*.bed")),
            separator="\t",
            has_header=False,
            # Read as strings: q-values such as "1" would otherwise be inferred as integers
            infer_schema=False,
            include_file_paths="path",
        )
        .group_by("path")
        .agg(
            # column_8 is the q-value of the DPR
            (pl.col("column_8").cast(pl.Float64) < q_threshold).any().alias("has_significant_dpr")
        )
        .with_columns(pl.col("path").str.extract(r"([^/]+)_intersecting_").alias("diff_dataset"))
        .drop("path")
        .collect()
    )

In [5]:
# ATAC-seq BED files are named differently from the query pairs.
# Pair-to-BED correspondence was confirmed by matching each job result (result_basename)
# to the renamed BED file by checksum.
atac_pair_to_dataset = pl.DataFrame(
    {
        "pair": [
            "pair1", "pair2", "pair3", "pair4", "pair5",
            "pair6", "pair7", "pair8", "pair10", "pair11",
        ],
        "diff_dataset": [
            "GSE101731_atac_hff_44C_60min",
            "GSE101731_atac_hff_44C_120min",
            "GSE154744_atac_k562_42C_12p5min_s",
            "GSE154744_atac_k562_42C_25min_s",
            "GSE154744_atac_k562_42C_40min_s",
            "GSE154744_atac_k562_42C_30min_hss",
            "GSE154744_atac_k562_42C_60min_hss",
            "GSE209675_mcf7_42C_60min",
            "GSE173536_atac_d21_42C_60min",
            "GSE173536_atac_t21_42C_60min",
        ],
    }
)

atac_dataset_metadata = (
    read_dpr_flag("../../out/ChIP-Atlas_diff/ATACseq/bedfile_intersect_v2")
    .join(atac_pair_to_dataset, on="diff_dataset", how="left")
    .join(read_srx_lists("../../config/chipatlas_diff/atacseq"), on="pair", how="left")
    .with_columns(
        # Placeholders to fill in manually
        pl.lit(None, dtype=pl.String).alias("sample"),
        pl.lit(None, dtype=pl.String).alias("temperature"),
        pl.col("pair").str.extract(r"(\d+)").cast(pl.Int64).alias("pair_number"),
    )
    .sort("pair_number")
    .select(
        "diff_dataset",
        "sample",
        # "temperature",
        "dataset_A_srx",
        "dataset_B_srx",
        "has_significant_dpr",
    )
)

print(f"ATAC-seq datasets with BED output: {atac_dataset_metadata.height}")
print(f"ATAC-seq datasets with significant DPR (q-value < 0.01): {atac_dataset_metadata['has_significant_dpr'].sum()}")
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(200)
display(atac_dataset_metadata)

ATAC-seq datasets with BED output: 10
ATAC-seq datasets with significant DPR (q-value < 0.01): 6


diff_dataset,sample,dataset_A_srx,dataset_B_srx,has_significant_dpr
str,str,str,str,bool
"""GSE101731_atac_hff_44C_60min""",null,"""SRX3025783,SRX3025784""","""SRX3025779,SRX3025780,SRX3025781,SRX3025782""",true
"""GSE101731_atac_hff_44C_120min""",null,"""SRX3025785,SRX3025786""","""SRX3025779,SRX3025780,SRX3025781,SRX3025782""",true
"""GSE154744_atac_k562_42C_12p5min_s""",null,"""SRX8772746,SRX8772747""","""SRX8772744,SRX8772745""",false
"""GSE154744_atac_k562_42C_25min_s""",null,"""SRX8772748,SRX8772749""","""SRX8772744,SRX8772745""",false
"""GSE154744_atac_k562_42C_40min_s""",null,"""SRX8772750,SRX8772751""","""SRX8772744,SRX8772745""",true
"""GSE154744_atac_k562_42C_30min_hss""",null,"""SRX8772736,SRX8772737""","""SRX8772734,SRX8772735""",false
"""GSE154744_atac_k562_42C_60min_hss""",null,"""SRX8772738,SRX8772739""","""SRX8772734,SRX8772735""",false
"""GSE209675_mcf7_42C_60min""",null,"""SRX16651805,SRX16651808""","""SRX16651804,SRX16651807""",true
"""GSE173536_atac_d21_42C_60min""",null,"""SRX10705939,SRX10705946,SRX10705947""","""SRX10705938,SRX10705944,SRX10705945""",true


In [6]:
# HSF1 BED files keep the pair number as a prefix (pairN_...), so no manual mapping is needed
hsf1_dataset_metadata = (
    read_dpr_flag("../../out/ChIP-Atlas_diff/ChIPseq_hsf1/bedfile_intersect_v2")
    .with_columns(pl.col("diff_dataset").str.extract(r"^(pair\d+)_").alias("pair"))
    .join(read_srx_lists("../../config/chipatlas_diff/chipseq_hsf1"), on="pair", how="left")
    .with_columns(
        # Placeholders to fill in manually
        pl.lit(None, dtype=pl.String).alias("sample"),
        pl.lit(None, dtype=pl.String).alias("temperature"),
        pl.col("pair").str.extract(r"(\d+)").cast(pl.Int64).alias("pair_number"),
    )
    .sort("pair_number")
    .select(
        "diff_dataset",
        "sample",
        # "temperature",
        "dataset_A_srx",
        "dataset_B_srx",
        "has_significant_dpr",
    )
)

print(f"HSF1 ChIP-seq datasets with BED output: {hsf1_dataset_metadata.height}")
print(f"HSF1 ChIP-seq datasets with significant DPR (q-value < 0.01): {hsf1_dataset_metadata['has_significant_dpr'].sum()}")
display(hsf1_dataset_metadata)

HSF1 ChIP-seq datasets with BED output: 6
HSF1 ChIP-seq datasets with significant DPR (q-value < 0.01): 6


diff_dataset,sample,dataset_A_srx,dataset_B_srx,has_significant_dpr
str,str,str,str,bool
"""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""",null,"""SRX155730,SRX155731""","""SRX155727,SRX155728""",true
"""pair2_GSE43579_hsf1_k562_hs_42C_30min""",null,"""SRX218220,SRX218221""","""SRX218217,SRX218218""",true
"""pair3_GSE43579_hsf1_k562_mitotic_hs_42C_30min""",null,"""SRX218227,SRX218228""","""SRX218224,SRX218225""",true
"""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""",null,"""SRX16646115,SRX16646116""","""SRX16646113,SRX16646114""",true
"""pair7_GSE270671_hsf1_pc3m_sijmjd6_hs_42C_60min""",null,"""SRX25380948,SRX25380949""","""SRX25380940,SRX25380941""",true
"""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""",null,"""SRX25380944,SRX25380945""","""SRX25380936,SRX25380937""",true


In [7]:
# Histone ChIP-seq dataset files have no pairN prefix.
# The dataset A file name equals the BED name (result_basename), and the A/B files of the same pair
# share the "GSEnnn_chip_<antibody>" prefix, which is used to pair them.
histone_srx_lists = (
    pl.scan_csv(
        sorted(pathlib.Path("../../config/chipatlas_diff/chipseq_histone").glob("dataset_*/*.txt")),
        has_header=False,
        new_columns=["srx_id"],
        include_file_paths="path",
    )
    .with_columns(
        pl.col("path").str.extract(r"(dataset_[AB])").alias("dataset"),
        pl.col("path").str.extract(r"/([^/]+)\.txt$").alias("file_stem"),
    )
    .with_columns(pl.col("file_stem").str.extract(r"^(GSE\d+_chip_[^_]+)_").alias("pair_key"))
    .with_columns(
        # Use the dataset A file name as diff_dataset for both A and B rows of the pair
        pl.col("file_stem").filter(pl.col("dataset") == "dataset_A").first().over("pair_key").alias("diff_dataset")
    )
    .group_by("diff_dataset", "dataset", maintain_order=True)
    .agg(pl.col("srx_id").str.join(","))
    .collect()
    .pivot(on="dataset", index="diff_dataset", values="srx_id")
    .rename({"dataset_A": "dataset_A_srx", "dataset_B": "dataset_B_srx"})
)

histone_dataset_metadata = (
    read_dpr_flag("../../out/ChIP-Atlas_diff/ChIPseq_histone/bedfile_intersect_v2")
    .join(histone_srx_lists, on="diff_dataset", how="left")
    .with_columns(
        # Placeholders to fill in manually
        pl.lit(None, dtype=pl.String).alias("sample"),
        pl.lit(None, dtype=pl.String).alias("temperature"),
    )
    .sort("diff_dataset")
    .select(
        "diff_dataset",
        "sample",
        # "temperature",
        "dataset_A_srx",
        "dataset_B_srx",
        "has_significant_dpr",
    )
)

print(f"Histone ChIP-seq datasets with BED output: {histone_dataset_metadata.height}")
print(f"Histone ChIP-seq datasets with significant DPR (q-value < 0.01): {histone_dataset_metadata['has_significant_dpr'].sum()}")
display(histone_dataset_metadata)

Histone ChIP-seq datasets with BED output: 2
Histone ChIP-seq datasets with significant DPR (q-value < 0.01): 2


diff_dataset,sample,dataset_A_srx,dataset_B_srx,has_significant_dpr
str,str,str,str,bool
"""GSE105028_chip_h3k27ac_hs_42C_60min""",null,"""SRX3288592,SRX3288590""","""SRX3288591,SRX3288589""",true
"""GSE209667_chip_h3k4me3_hs_42C_60min""",null,"""SRX16646109,SRX16646092""","""SRX16646108,SRX16646110""",true


In [8]:
# Save both tables as sheets of one Excel file for pasting into Word
with xlsxwriter.Workbook("../../out/figshare/chip_atlas_diff_analysis/diff_analysis_dataset_metadata.xlsx") as wb:
    atac_dataset_metadata.write_excel(wb, worksheet="ATAC-seq", autofit=True)
    hsf1_dataset_metadata.write_excel(wb, worksheet="HSF1 ChIP-seq", autofit=True)
    histone_dataset_metadata.write_excel(wb, worksheet="Histone ChIP-seq", autofit=True)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1-2. Metadata Filtering

- [metadata process notebook](../00_metadata/) では、Diff Analysis に使われたかどうかでフィルタリングをしておらず、とりあえず使えそうなデータを記述している
- そこから、上記の表に基づいて、Diff Analysis に使ったデータだけの表を作成し、figshare にアップロードする

&nbsp;

### 1-2-1. ATAC-seq

In [9]:
atacseq_all_curated_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/atacseq_metadata_all_261001.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(2)
display(atacseq_all_curated_metadata)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_series_id,cell_line,genome_build,sample_accession,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,is_chip_atlas_registered,library_selection,secondary_study_accession,study_title,description,cell_type,library_strategy,library_source,library_layout,library_construction_protocol,experiment_alias,center_name,scientific_name,tax_id,study_accession,secondary_sample_accession,treatment,stress_with,replicate,sample_characteristics_ch1,preconditioning,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str
"""SRX3316069""","""SRX3316067""","""hs_fast_atac-seq_rep1 vs fast_atac-seq_rep1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6206775 vs SRR6206773""","""HS Fast ATAC-Seq Rep1 vs Fast ATAC-Seq Rep1""","""GSM2829008 vs GSM2829006""","""Public on Aug 03 2018""","""Oct 24 2017""","""May 15 2019""","""SRA""","""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""fast ATAC-Seq, Fast ATAC-Seq under heat shock conditions, HS_ATAC-Seq.bw vs fast ATAC-Seq, Fast ATAC-Seq under normal conditions, ATAC-Seq.bw""","""GPL11154""","""Illumina HiSeq 2000""","""other""","""genomic""","""ATAC-seq""","""GSM2829008 vs GSM2829006""","""GSE105028""","""hESC""","""hg38""","""SAMN07828432 vs SAMN07828392""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""42 C""","""60 min.""","""37 C (inferred)""",false,"""other""","""SRP120027""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""Illumina HiSeq 2000 sequencing: GSM2829008: HS Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq vs Illumina HiSeq 2000 sequencing: GSM2829006: Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq""","""H9 human Embryonic Stem Cell Line""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""For Hi-C, cells were crosslinked with 1% formaldehyde for 10 min at room temperature and quenched with glycine, nuclei were isolated and chromatin digested by HindIII, filled in with biotin-dCTP, and …","""GSM2829008 vs GSM2829006""","""GEO""","""Homo sapiens""",9606,"""PRJNA414442""","""SRS2621656 vs SRS2621654""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX10705951""","""SRX10705949""","""atacseq_lymphblastoid_cells_t21_42c_rep2_technical2 vs atacseq_lymphblastoid_cells_t21_37c_rep2_technical2""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR14352104 vs SRR14352102""","""ATACseq_Lymphblastoid cells_T21_42C_rep2_technical2 vs ATACseq_Lymphblastoid cells_T21_37C_rep2_technical2""","""GSM5269436 vs GSM5269434""","""Public on Apr 21 2023""","""Apr 28 2021""","""Apr 21 2023""","""SRA""","""lymphoblastoid-EBV immortalized""","""Homo sapiens""",9606,"""genomic DNA""","""Ethan_T21, genomic DNA (ATAC-seq), Ethan42merged_peaks_clean.narrowPeak vs Ethan_T21, genomic DNA (ATAC-seq), Ethan37merged_peaks_clean.narrowPeak""","""GPL18573""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM5269436 vs GSM5269434""","""GSE173536""",null,"""hg38 (GRCh38)""","""SAMN18915517 vs SAMN18915519""","""https://doi.org/10.1371/journal.pone.0307375""","""42 C""","""60 min.""","""37 C (inferred)""",true,"""other""","""SRP316796""","""Blood cells with trisomy 21 overreact to heat shock stress [ATAC-seq]""","""NextSeq 500 sequencing: GSM5269436: ATACseq_L

In [10]:
# Keep only heat/control sample pairs used in the Diff Analysis runs that produced a BED file.
# A metadata row is kept when its heat SRX is in dataset A and its control SRX is in dataset B
# of the same diff_dataset.

# 左に寄せたいカラムリスト
# Columns to move to the left, in this order
LEFT_COLUMNS = ["diff_dataset", "sample_heat_id", "sample_control_id", "sample_name", "cell_type", "stress treatment temperature","stress treatment time", "control treatment temperature","reference_doi"]

atacseq_metadata_diff = (
    atacseq_all_curated_metadata
    .join(
        atac_dataset_metadata.select(
            "diff_dataset",
            pl.col("dataset_A_srx").str.split(",").alias("sample_heat_id"),
            pl.col("dataset_B_srx").str.split(","),
        ).explode("sample_heat_id"),
        on="sample_heat_id",
        how="inner",
    )
    .filter(
        pl.col("dataset_B_srx").list.contains(pl.col("sample_control_id"))
    )
    .drop("dataset_B_srx")
    .select("diff_dataset", pl.exclude("diff_dataset"))
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS) # 左に寄せたいカラム
).drop(
    "genome_build",
    "protocol",
    "scientific_name",
    "tax_id",
    "center_name",
    "sample_organism_ch1",
    "is_chip_atlas_registered"
)

print(f"metadata rows: {atacseq_all_curated_metadata.height} -> {atacseq_metadata_diff.height}")
print(f"diff_dataset count: {atacseq_metadata_diff.select('diff_dataset').n_unique()}")
display(atacseq_metadata_diff)

metadata rows: 34 -> 26
diff_dataset count: 10


diff_dataset,sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_source_name_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_series_id,cell_line,sample_accession,library_selection,secondary_study_accession,study_title,description,library_strategy,library_source,library_layout,library_construction_protocol,experiment_alias,study_accession,secondary_sample_accession,treatment,stress_with,replicate,sample_characteristics_ch1,preconditioning,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""GSE101731_atac_hff_44C_60min""","""SRX3025783""","""SRX3025779""","""1h_heat_stress_rep_1 vs mock_rep_1""","""Primary human foreskin fibroblasts (HFF)""","""44 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1371/journal.ppat.1006954""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR5856839 vs SRR5856835""","""1h heat stress Rep 1 vs Mock Rep 1""","""GSM2714164 vs GSM2714160""","""Public on Mar 08 2018""","""Jul 21 2017""","""May 15 2019""","""SRA""","""HFF_1h_44ºC_ATAC-seq vs HFF_mock_ATAC-seq""",9606,"""genomic DNA""","""1h heat stress Rep 1 vs Mock Rep 1""","""GPL18573""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM2714164 vs GSM2714160""","""GSE101731""",null,"""SAMN07375245 vs SAMN07375249""","""other""","""SRP113284""","""ATAC-seq of HFF exposed to salt and heat stress""","""NextSeq 500 sequencing: GSM2714164: 1h heat stress Rep 1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM2714160: Mock Rep 1 Homo sapiens ATAC-seq""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""Samples were processed according to the original ATAC-seq protocol (Buenrostro et al. Nature Methods 2013) See Buenrostro et al, Nature Methods 2013 NextSeq 500 sequencing 75bp paired-end reads""","""GSM2714164 vs GSM2714160""","""PRJNA395312""","""SRS2374866 vs SRS2374862""",null,"""44ºC vs mock""","""Repicate 1""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GSE173536_atac_t21_42C_60min""","""SRX10705951""","""SRX10705949""","""atacseq_lymphblastoid_cells_t21_42c_rep2_technical2 vs atacseq_lymphblastoid_cells_t21_37c_rep2_technical2""","""lymphoblastoid""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1371/journal.pone.0307375""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR14352104 vs SRR14352102""","""ATACseq_Lymphblastoid cells_T21_42C_rep2_technical2 vs ATACseq_Lymphblastoid cells_T21_37C_rep2_technical2""","""GSM5269436 vs GSM5269434""","""Public on Apr 21 2023""","""Apr 28 2021""","""Apr 21 2023""","""SRA""","""lymphoblastoid-EBV immortalized""",9606,"""genomic DNA""","""Ethan_T21, genomic DNA (ATAC-seq), Ethan42merged_peaks_clean.narrowPeak vs Ethan_T21, genomic DNA (ATAC-seq), Ethan37merged_peaks_clean.narrowPeak""","""GPL18573""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM5269436 vs GSM5269434""","""GSE173536""",null,"""SAMN18915517 vs SAMN18915519""","""other""","""SRP316796""","""Blood cells with trisomy 21 overreact to heat shock stress [ATAC-seq]""","""NextSeq 500 sequencing: GSM5269436: ATACseq_Lymphblastoid cells_T21_42C_rep2_technical2 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM5269434: ATACseq_Lymphblastoid cells_T21_37C_rep2_technical2…","""ATAC-seq""","""GENOMIC""","""PAIRED""","""Suspension cells were then used to prepare an ATAC-seq library using the OMNI-ATAC-seq protocol. For control samples in calling ATAC-seq peaks, genomi

In [11]:
# For figshare
atacseq_metadata_diff.write_csv(
    "../../out/figshare/chip_atlas_diff_analysis/diff_analysis_dataset_metadata/hs_related_atacseq_metadata_diff_analysis_pair.tsv",
    separator="\t"
)

&nbsp;

### 1-2-2. ChIP-seq (HSF1)

In [12]:
chipseq_all_curated_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/chipseq_metadata_all_261001.tsv",
    separator="\t",
    schema_overrides={"sample_contact_zip_postal_code": pl.Utf8}
)

pl.Config.set_tbl_rows(2)
display(chipseq_all_curated_metadata)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,…,tissue_cells,age,fastq_galaxy,fastq_ftp,fastq_aspera,library_strategy_right,sra_file_role,sra_aspera,sra_galaxy,sra_ftp,sample_characteristics_ch1,tumor,passages,country,collection_date,assembly,tissue,tissue_type,chip,basecalling,adapter_trimming,align,deduplication,peak_calling,batch,library_name_right,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,broker_name,datahub,submitted_file_role,submitted_read_type,submitted_format
str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX7107227""","""SRX7107225""","""vcap_pol2_sictrl_hs1 vs vcap_pol2_sictrl_ctrl1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""VCaP: Pol2_siCtrl_HS1 vs VCaP: Pol2_siCtrl_Ctrl1""","""GSM4152129 vs GSM4152127""","""Public on Feb 05 2021""","""Nov 06 2019""","""Feb 06 2021""","""SRA""",1,"""VCaP""","""Homo sapiens""",9606,"""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min. Silencing for VCaP cells was done by transfecting siRNA for UBE2I (siUBE2I samples) or non-targeting control siRNA (siNON s…","""VCaP cells were grown in DMEM medium supplemented with 10% feta bovine serum and 1% penicillin/streptomycin antibiotics at 37 degree C temperature incubator with 5% CO2.""","""genomic DNA""","""VCaP cells were grown in DMEM medium supplemented with 10% feta bovine serum and 1% penicillin/streptomycin antibiotics at 37 degree C temperature incubator with 5% CO2. Nuclei from approximately 10 m…","""resequencing of sample previously submitted to GEO as GSM1777590 vs resequencing of sample previously submitted to GEO as GSM1777588""","""ChIP-seq reads were aligned to the human genome build hg38 using Bowtie2. Duplicate and discordant reads were removed. Gene annotations and transcript start site (TSS) information for human genes wer…","""GPL18573""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4152129 vs GSM4152127""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4152nnn/GSM4152129/suppl/GSM4152129_Sample3_Pol2_siCtrl_HS1.filtered.seq_depth_norm.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4152nnn/GSM4152127/suppl/G…","""GSE140053""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ERX432311""","""ERX432313""",null,null,null,null,null,"""ERR466074 vs ERR466075""","""growing_U2OS_cells_4 vs growing_U2OS_cells_3""",null,null,null,null,null,null,null,null,null,null,null,null,null,"""Protocols: Human osteosarcoma (U2OS) cells were obtained from CRUK and grown at 37C according to standard cell culture protocols. U2OS cells were grown at 37C and either left untreated or incubated at…",null,null,null,null,null,null,null,null,null,null,nu

In [13]:
# Keep only heat/control sample pairs used in the HSF1 Diff Analysis runs that produced a BED file.
# A metadata row is kept when its heat SRX is in dataset A and its control SRX is in dataset B
# of the same diff_dataset.

# Columns to move to the left, in this order
LEFT_COLUMNS = ["diff_dataset", "sample_heat_id", "sample_control_id", "sample_name", "cell_type", "stress treatment temperature","stress treatment time", "control treatment temperature","reference_doi"]

chipseq_hsf1_metadata_diff = (
    chipseq_all_curated_metadata
    .join(
        hsf1_dataset_metadata.select(
            "diff_dataset",
            pl.col("dataset_A_srx").str.split(",").alias("sample_heat_id"),
            pl.col("dataset_B_srx").str.split(","),
        ).explode("sample_heat_id"),
        on="sample_heat_id",
        how="inner",
    )
    .filter(
        pl.col("dataset_B_srx").list.contains(pl.col("sample_control_id"))
    )
    .drop("dataset_B_srx")
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS)
).drop(
    # is_chip_atlas_registered does not exist in the ChIP-seq metadata
    "genome_build",
    "protocol",
    "scientific_name",
    "tax_id",
    "center_name",
    "sample_organism_ch1",
    "sample_channel_count",
    "fastq_galaxy",
    "fastq_ftp",
    "fastq_aspera",
    "library_strategy_right",
    "sra_file_role",
    "sra_aspera",
    "sra_galaxy",
    "sra_ftp",
    "sample_characteristics_ch1",
    "tumor",
    "passages",
    "country",
    "collection_date",
    "assembly",
    "tissue",
    "tissue_type",
    "chip",
    "basecalling",
    "adapter_trimming",
    "align",
    "deduplication",
    "peak_calling",
    "batch",
    "library_name_right",
    "submitted_galaxy",
    "submitted_ftp",
    "submitted_aspera",
    "submitted_md5",
    "checklist",
    "submitted_bytes",
    "broker_name",
    "datahub",
    "submitted_file_role",
    "submitted_read_type",
    "submitted_format"
)

print(f"metadata rows: {chipseq_all_curated_metadata.height} -> {chipseq_hsf1_metadata_diff.height}")
print(f"diff_dataset count: {chipseq_hsf1_metadata_diff.select('diff_dataset').n_unique()}")
display(chipseq_hsf1_metadata_diff)

metadata rows: 217 -> 12
diff_dataset count: 6


diff_dataset,sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_source_name_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,…,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias,sample_contact_institute,antibody,tumorigenic_or_nontumorigenic,42˚_c_1h_heat-shock_,fastq_file_role,cell_type_right,tax_lineage,library_name,tag,sample_contact_phone,sample_supplementary_file_2,cell_cycle_phase,antibody_manufacturer,supplementary_files_format_and_content,treatment1,processed_data_file,sex,tissue_cells,age
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""SRX155730""","""SRX155727""","""mcf10a_hs_hsf1_chipseq_1 vs mcf10a_hsf1_chipseq_1""","""Immortalized mammary epithelial cells""","""42 C""","""60 min.""",null,"""https://doi.org/10.1016/j.cell.2012.06.031""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR515405 vs SRR515402""","""MCF10A_HS_HSF1_ChIPSeq_1 vs MCF10A_HSF1_ChIPSeq_1""","""GSM951864 vs GSM951861""","""Public on Aug 07 2012""","""Jun 24 2012""","""May 15 2019""","""SRA""","""MCF10A CELLS""",9606,null,null,"""genomic DNA""",null,"""MCF10A_HS_HSF1_ChIPSeq_1 vs MCF10A_HSF1_ChIPSeq_1""",null,"""GPL11154""","""Marc,,Mendillo""","""9 Cambridge Center""","""Cambridge""","""2142""","""USA""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""",…,"""Illumina HiSeq 2000 sequencing: GSM951864: MCF10A_HS_HSF1_ChIPSeq_1 Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM951861: MCF10A_HSF1_ChIPSeq_1 Homo sapiens ChIP-Seq""",null,"""522492800 vs 1655846160""","""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""GSM951864 vs GSM951861""","""GSM951864_r1 vs GSM951861_r1""",null,"""GSM951864_1 vs GSM951861_1""",null,"""2013-11-11""","""PRJNA169333""","""SRR515405 vs SRR515402""","""SRS346799 vs SRS346796""","""2012-08-31""","""GSE38901""","""Whitehead Institute for Biomedical Research""","""HSF1 (H-311, Santa Cruz, sc-9144, Lot# J0510)""","""Non-tumorigenic""","""YES vs No""","""GENERATED_FILE""","""Immortalized mammary epithelial cells""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""GSM951864: MCF10A_HS_HSF1_ChIPSeq_1 vs GSM951861: MCF10A_HSF1_ChIPSeq_1""","""env_tax:terrestrial;env_tax""",null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""","""SRX25380945""","""SRX25380937""","""pc3m_sint_hs_hsf1_chip_biol_rep_2 vs pc3m_sint_basal_hsf1_chip_biol_rep_2""","""Prostate cancer cell line""","""42 C""","""60 min.""",null,"""https://doi.org/10.1073/pnas.2313370121""",null,null,null,"""SRR29884645 vs SRR29884680""","""PC3M, siNT, HS, HSF1 ChIP, biol rep 2 vs PC3M, siNT, Basal, HSF1

In [14]:
# For figshare
chipseq_hsf1_metadata_diff.write_csv(
    "../../out/figshare/chip_atlas_diff_analysis/diff_analysis_dataset_metadata/hs_related_chipseq_hsf1_metadata_diff_analysis_pair.tsv",
    separator="\t"
)

&nbsp;

### 1-2-3. ChIP-seq (histone modification)

In [15]:
chipseq_histone_all_curated_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/chipseq_histone_modification_metadata_all_261001.tsv",
    separator="\t",
)

pl.Config.set_tbl_rows(2)
display(chipseq_histone_all_curated_metadata)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,…,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,sample_contact_state,condition,sample_description_right,cell_type,library_strategy_right,center_name,sample_data_processing,chip_antibody,assembly,country,cell_type_right,collection_date,library_name,basecalling,adapter_trimming,align,deduplication,peak_calling,supplementary_files_format_and_content
str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX2313416""","""SRX2313415""","""h4ac_k562_hs30 vs h4ac_k562_nhs""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR4779641 vs SRR4779640""","""H4ac K562 HS30 vs H4ac K562 NHS""","""GSM2367734 vs GSM2367733""","""Public on May 21 2017""","""Nov 01 2016""","""May 15 2019""","""SRA""",1,"""K562 cells""","""Homo sapiens""",9606,"""genomic DNA""","""GPL16791""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina HiSeq 2500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2367734 vs GSM2367733""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2367nnn/GSM2367734/suppl/GSM2367734_H4ac_ChIP-seq_K562_UpState06-866_R1_36nt_HS30.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2367nnn/GSM2367733/suppl…","""GSE89382""",…,"""PAIRED""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR477/001/SRR4779641/SRR4779641_1.fastq.gz;fasp.sra.ebi.ac.uk:/vol1/fastq/SRR477/001/SRR4779641/SRR4779641_2.fastq.gz vs fasp.sra.ebi.ac.uk:/vol1/fastq/SRR477/000/SRR47…","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""GSM2367734 vs GSM2367733""","""GSM2367734_r1 vs GSM2367733_r1""","""After tretments, the cells were cross-linked with 1% fromaldehyde, quenched with glycine, washed twice with PBS and flash-frozen. The ChIP-experiments were conducted as described for GSE43579 (Viherva…","""GSM2367734 vs GSM2367733""","""K562""","""env_tax:terrestrial;env_tax""","""2017-05-23""","""Homo sapiens""",9606,"""PRJNA352038""","""SRR4779641 vs SRR4779640""","""SRS1770826 vs SRS1770824""","""2017-05-23""","""GSE89382""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX25966379""","""SRX25966376""","""k27ac_hs_22rv1_h1ko vs k27ac_nhs_22rv1_h1ko""",null,null,null,null,"""SRR30542967 vs SRR30542970""","""K27ac_HS_22RV1_H1KO vs K27ac_NHS_22RV1_H1KO""","""GSM8496182 vs GSM8496179""","""Public on Sep 02 2025""","""Sep 04 2024""","""Sep 02 2025""","""SRA""",1,"""22RV1""","""Homo sapiens""",9606,"""genomic DNA""","""GPL2

In [16]:
# Keep only heat/control sample pairs used in the histone Diff Analysis runs that produced a BED file.
# A metadata row is kept when its heat SRX is in dataset A and its control SRX is in dataset B
# of the same diff_dataset.

# Columns to move to the left, in this order
# ("control treatment temperature" does not exist in the histone metadata)
LEFT_COLUMNS = ["diff_dataset", "sample_heat_id", "sample_control_id", "sample_name", "cell_type", "stress treatment temperature","stress treatment time", "reference_doi"]

chipseq_histone_metadata_diff = (
    chipseq_histone_all_curated_metadata
    .join(
        histone_dataset_metadata.select(
            "diff_dataset",
            pl.col("dataset_A_srx").str.split(",").alias("sample_heat_id"),
            pl.col("dataset_B_srx").str.split(","),
        ).explode("sample_heat_id"),
        on="sample_heat_id",
        how="inner",
    )
    .filter(
        pl.col("dataset_B_srx").list.contains(pl.col("sample_control_id"))
    )
    .drop("dataset_B_srx")
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS)
).drop(
    # Only columns that exist in the histone metadata
    "genome_build",
    "protocol",
    "scientific_name",
    "tax_id",
    "center_name",
    "sample_organism_ch1",
    "sample_channel_count",
    "fastq_galaxy",
    "fastq_ftp",
    "fastq_aspera",
    "library_strategy_right",
    "country",
    "collection_date",
    "assembly",
    "basecalling",
    "adapter_trimming",
    "align",
    "deduplication",
    "peak_calling",
)

print(f"metadata rows: {chipseq_histone_all_curated_metadata.height} -> {chipseq_histone_metadata_diff.height}")
print(f"diff_dataset count: {chipseq_histone_metadata_diff.select('diff_dataset').n_unique()}")
display(chipseq_histone_metadata_diff)

metadata rows: 12 -> 4
diff_dataset count: 2


diff_dataset,sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,reference_doi,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_source_name_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,…,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,tag,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias,sample_contact_state,condition,sample_description_right,sample_data_processing,chip_antibody,cell_type_right,library_name,supplementary_files_format_and_content
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""GSE105028_chip_h3k27ac_hs_42C_60min""","""SRX3288592""","""SRX3288591""","""hs_h3k27ac_chip-seq_rep vs h3k27ac_chip-seq_rep""","""null""","""42 C""","""60 min.""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177970 vs SRR6177969""","""HS H3K27ac ChIP-Seq Rep vs H3K27ac ChIP-Seq Rep""","""GSM2816651 vs GSM2816650""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""","""H9 hESCs""",9606,"""genomic DNA""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""",null,null,"""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""361102""","""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2816651 vs GSM2816650""",…,"""ILLUMINA""","""Generic""","""ChIP-Seq under heat shock conditions, H3K27ac_HS.bw vs ChIP-Seq under normal conditions, H3K27ac_NT.bw""","""a69f7807d1925c69015c7eba7dbf1a99;5e0f7dfff958e72f28c1df02126d7b96 vs d2195ff1cf36ceba13499627b9775eda;c252c1c43462446d9c8e1b073cc64efd""","""SRP120027""","""8548663 vs 10283229""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""GENERATED_FILE;GENERATED_FILE""","""HS H3K27ac ChIP-Seq Rep vs H3K27ac ChIP-Seq Rep""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2816651: HS H3K27ac ChIP-Seq Rep Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM2816650: H3K27ac ChIP-Seq Rep Homo sapiens ChIP-Seq""","""854866300 vs 1028322900""",null,"""GENOMIC""","""PAIRED""","""public""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""GSM2816651 vs GSM2816650""","""GSM2816651_r1 vs GSM2816650_r1""","""For Hi-C, cells were crosslinked with 1% formaldehyde for 10 min at room temperature and quenched with glycine, nuclei were isolated and chromatin digested by HindIII, filled in with biotin-dCTP, and …","""GSM2816651 vs GSM2816650""","""H9 human Embryonic Stem Cell Line""","""env_tax:terrestrial;env_tax""","""2018-08-04""","""PRJNA414442""","""SRR6177970 vs SRR6177969""","""SRS2598128 vs SRS2598127""","""2018-08-04""","""GSE105028""","""FUJIAN""","""under heat shock conditions vs under norm

In [17]:
# For figshare
chipseq_histone_metadata_diff.write_csv(
    "../../out/figshare/chip_atlas_diff_analysis/diff_analysis_dataset_metadata/hs_related_chipseq_histone_metadata_diff_analysis_pair.tsv",
    separator="\t"
)